# EVOPS Tier-2: CLIP GPU Evaluation

Kaggle Notebook for scoring EVOPS images with OpenAI CLIP ViT-B/32. Set the accelerator to GPU T4, enable Internet for first-time model/package downloads, and attach a Dataset containing `evops_<PAPER_ID>_<CONCEPT>.zip`.

Outputs: `scores.csv`, `summary.csv`, and optionally `clip_hist.png` under `/kaggle/working`.

**Interpretation:** CLIP similarity is a prompt-relative alignment signal, not a calibrated image-quality score or human rating. Compare runs only with consistent prompts, model, and preprocessing.

In [ ]:
# ===== EVOPS Tier-2: CLIP GPU Evaluation =====
# Runtime: GPU T4 / Internet ON / attach Dataset containing the input ZIP

import os
from pathlib import Path

# ---- User configuration ----
PAPER_ID = "S01"
CONCEPT = "donpen"
ZIP_NAME = f"evops_{PAPER_ID}_{CONCEPT}.zip"
TEXT_PROMPTS = [
    f"{CONCEPT}, cute character",
    f"high quality illustration of {CONCEPT}",
]

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/tmp/evops")
OUT_DIR = Path("/kaggle/working")
OUT_CSV = OUT_DIR / "scores.csv"
SUMMARY_CSV = OUT_DIR / "summary.csv"

# Optional Hugging Face secret; OpenAI CLIP ViT-B/32 itself does not require HF_TOKEN.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as exc:
    print("HF_TOKEN unavailable; continuing (not required for public OpenAI CLIP):", type(exc).__name__)

print("PAPER_ID =", PAPER_ID, "| CONCEPT =", CONCEPT)
print("ZIP_NAME =", ZIP_NAME)
print("TEXT_PROMPTS =", TEXT_PROMPTS)

In [ ]:
# Install CLIP and its lightweight dependencies
!pip -q install ftfy regex tqdm
!pip -q install git+https://github.com/openai/CLIP.git

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is unavailable; evaluation will run on CPU.")

In [ ]:
import zipfile
import shutil

WORK_DIR.mkdir(parents=True, exist_ok=True)
zip_candidates = sorted(INPUT_ROOT.rglob(ZIP_NAME))
if not zip_candidates:
    zip_candidates = sorted(INPUT_ROOT.rglob("evops_*.zip"))
if not zip_candidates:
    raise FileNotFoundError(
        f"{ZIP_NAME} not found. Attach the Kaggle Dataset containing the ZIP; "
        "check the name or upload it with kaggle_push."
    )

zip_path = zip_candidates[0]
print("Using ZIP:", zip_path)
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

# Dataset ZIPs should be trusted. Avoid extracting untrusted archives.
with zipfile.ZipFile(zip_path, "r") as zf:
    for member in zf.infolist():
        target = (WORK_DIR / member.filename).resolve()
        if target != WORK_DIR.resolve() and WORK_DIR.resolve() not in target.parents:
            raise ValueError(f"Unsafe ZIP path: {member.filename}")
    zf.extractall(WORK_DIR)

for p in sorted(WORK_DIR.rglob("*"))[:30]:
    print(p.relative_to(WORK_DIR))

In [ ]:
from PIL import Image

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def collect_images(root: Path):
    return [p for p in sorted(root.rglob("*")) if p.is_file() and p.suffix.lower() in IMAGE_EXTS]

image_paths = collect_images(WORK_DIR)
print(f"Found {len(image_paths)} images")

if len(image_paths) == 0:
    video_paths = sorted(list(WORK_DIR.rglob("*.mp4")) + list(WORK_DIR.rglob("*.webm")))
    if video_paths:
        import cv2
        video_path = video_paths[0]
        print("No still images; extracting frames from", video_path)
        frames_dir = WORK_DIR / "frames"
        frames_dir.mkdir(exist_ok=True)
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 24
        step = max(1, int(fps))
        i = saved = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if i % step == 0:
                frame_path = frames_dir / f"frame_{saved:05d}.png"
                cv2.imwrite(str(frame_path), frame)
                saved += 1
            i += 1
        cap.release()
        image_paths = collect_images(frames_dir)
        print(f"Extracted {len(image_paths)} frames")
    else:
        raise FileNotFoundError("No images or supported videos were found in the input ZIP.")

print("Sample:", image_paths[0] if image_paths else None)

In [ ]:
import clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

text_tokens = clip.tokenize(TEXT_PROMPTS).to(device)
with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print("Model: ViT-B/32 | device =", device)
print("Text prompts:", TEXT_PROMPTS)

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

rows = []
with torch.no_grad():
    for path in tqdm(image_paths, desc="CLIP scoring"):
        try:
            with Image.open(path) as im:
                image = preprocess(im.convert("RGB")).unsqueeze(0).to(device)
            image_features = model.encode_image(image)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            sims = (image_features @ text_features.T).squeeze(0).cpu().numpy()
            clip_score = float(np.mean(sims))
            # Convenience transform only; this is NOT a calibrated quality score.
            clip_score_100 = (clip_score + 1.0) * 50.0
            rows.append({
                "paper_id": PAPER_ID,
                "concept": CONCEPT,
                "path": str(path.relative_to(WORK_DIR)),
                "clip_score": round(clip_score, 6),
                "clip_score_100": round(clip_score_100, 2),
                "prompt_sims": ";".join(f"{float(s):.4f}" for s in sims),
                "status": "ok",
            })
        except Exception as exc:
            rows.append({
                "paper_id": PAPER_ID,
                "concept": CONCEPT,
                "path": str(path.relative_to(WORK_DIR)),
                "clip_score": None,
                "clip_score_100": None,
                "prompt_sims": "",
                "status": f"ERROR: {type(exc).__name__}: {exc}",
            })

df = pd.DataFrame(rows)
display(df.head(10))
print("Rows:", len(df))
print("Successful:", int(df["clip_score"].notna().sum()))
print("Mean CLIP similarity:", df["clip_score"].mean())
print("Mean transformed score:", df["clip_score_100"].mean())

In [ ]:
valid = df["clip_score"].dropna()
valid_100 = df["clip_score_100"].dropna()
summary = {
    "paper_id": PAPER_ID,
    "concept": CONCEPT,
    "model": "openai/CLIP ViT-B/32",
    "device": device,
    "prompts": " || ".join(TEXT_PROMPTS),
    "n_images": int(valid.size),
    "n_errors": int(df["clip_score"].isna().sum()),
    "mean_clip": float(valid.mean()) if valid.size else None,
    "mean_clip_100": float(valid_100.mean()) if valid_100.size else None,
    "min_clip_100": float(valid_100.min()) if valid_100.size else None,
    "max_clip_100": float(valid_100.max()) if valid_100.size else None,
}
summary_df = pd.DataFrame([summary])
display(summary_df)

OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CSV, index=False)
summary_df.to_csv(SUMMARY_CSV, index=False)
print("Wrote:", OUT_CSV)
print("Wrote:", SUMMARY_CSV)
print("Download these files from the Notebook Output or retrieve them with `kaggle kernels output`.")

In [ ]:
import matplotlib.pyplot as plt

valid = df["clip_score_100"].dropna()
if len(valid):
    plt.figure(figsize=(8, 4))
    plt.hist(valid, bins=20, edgecolor="black", alpha=0.8)
    plt.axvline(valid.mean(), linestyle="--", label=f"mean={valid.mean():.1f}")
    plt.xlabel("Transformed CLIP similarity (0–100; not calibrated quality)")
    plt.ylabel("count")
    plt.title(f"CLIP scores — {PAPER_ID} / {CONCEPT}")
    plt.legend()
    plt.tight_layout()
    hist_path = OUT_DIR / "clip_hist.png"
    plt.savefig(hist_path, dpi=120)
    plt.show()
    print("Wrote:", hist_path)
else:
    print("No valid scores to plot.")

## EVOPS integration notes

- Import `scores.csv` and `summary.csv` into the local EVOPS evaluation pipeline; keep the original CSVs as immutable experiment artifacts.
- Record the Kaggle dataset version, notebook version/commit, prompt list, CLIP model, and date with each experiment.
- Do not treat `clip_score_100` as a human score. A weighted blend such as `local_score * 0.4 + clip_score_100 * 0.6` is only meaningful after calibration on held-out human-rated data.
- Keep API tokens and private or personally identifying data out of the uploaded Dataset.